## 02 - Data Cleaning

In [27]:
import pandas as pd
import numpy as np

In [28]:
df=pd.read_csv("../data/processed/online_retail_II_processed.csv")

In [29]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str')

In [30]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

### Column Renaming

In [31]:
df=df.rename(columns={
    'Invoice':'invoice_number', 
    'StockCode':'stock_code',
    'Description':'description',
    'Quantity':'quantity',
    'InvoiceDate':'invoice_date',
    'Price':'unit_price',
    'Customer ID':'customer_id',
    'Country':'country'
})

### Normalizing Datatypes

In [32]:
df=df.astype({
    'invoice_number':'string', 
    'stock_code':'string',
    'description':'string',
    'quantity':'Int64',
    'invoice_date':'datetime64[ns]',
    'unit_price':'float64',
    'customer_id':"Int64",
    'country':'string'
})

### Removing exact duplicates

In [33]:
df.duplicated().sum()

np.int64(34335)

In [34]:
df = df.drop_duplicates(keep="first")

In [35]:
df.shape

(1033036, 8)

### Normalize cancelled positive quantities

In [36]:
df.loc[(df["invoice_number"].str.startswith("C")) & (df["quantity"]>0), "quantity"] *=-1

### Normalize the Stockcode

In [37]:
df["stock_code"] = df["stock_code"].str.upper()

In [ ]:
df["description"] = df["description"].str.strip()
df["stock_code"] = df["stock_code"].str.strip()
df["invoice_number"] = df["invoice_number"].str.strip()
df["country"] = df["country"].str.strip()

### Creating and Classify transaction status

In [38]:
conditions=[
    df["invoice_number"].str.startswith("C",na=False),
    df["quantity"]<0
]
choices=[
    "Cancelled",
    "Return/Adjusted"
]

df["transaction_status"]= np.select(
    conditions,
    choices,
    default="Completed"
)

### Excluding invalid/test records

In [39]:
remove_codes=[
    "TEST001",
    "TEST002",
    "GIFT",
    "DCGSLGIRL",
    "DCGSLBOY"
]
df=df[~df["stock_code"].isin(remove_codes)]

### Extracting Date, year, month, and Time from invoice date

In [40]:
df["transaction_date"]=df["invoice_date"].dt.date

In [41]:
df["transaction_month"]=df["invoice_date"].dt.month

In [42]:
df["transaction_year"]=df["invoice_date"].dt.year

In [43]:
df["transaction_time"]=df["invoice_date"].dt.time

### Calculating revenue fields

In [44]:
df["gross_revenue"]=np.where(
    df["quantity"]>0,
    df['quantity']*df['unit_price'],
    0
)

In [45]:
df['return_value']=np.where(
    df['quantity']<0,
    abs(df['quantity']*df['unit_price']),
    0
)

In [46]:
df['net_revenue']=df['gross_revenue']-df['return_value']

In [47]:
df=df[
    [
        'invoice_number',
        'stock_code',
        'description',
        'quantity',
        'invoice_date',
        'unit_price',
        'customer_id',
        'country',
        'transaction_date',
        'transaction_month',
        'transaction_year',
        'transaction_time',
        'gross_revenue',
        'return_value',
        'net_revenue',
        'transaction_status'
    ]
]

### Saving Dataset

In [48]:
df.to_csv("../data/cleaned/online_retail_II_cleaned.csv", index=False)